In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/Porfolio/Balearia/2 Demanda"

# Creamos estructura del proyecto
os.makedirs(f"{PROJECT_ROOT}/data/raw", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/data/processed", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/reports/figures", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/reports/model_cards", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/dashboards/powerbi_mockups", exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Exists?:", os.path.exists(PROJECT_ROOT))
print("Processed folder:", os.path.exists(f"{PROJECT_ROOT}/data/processed"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/Porfolio/Balearia/2 Demanda
Exists?: True
Processed folder: True


### Descargar `trips_df` como CSV

In [15]:
from google.colab import files

# Convertir trips_df a CSV
trips_df.to_csv('trips.csv', index=False)

# Descargar el archivo
files.download('trips.csv')

print("Archivo 'trips.csv' generado y listo para descargar.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Archivo 'trips.csv' generado y listo para descargar.


### Descargar `bookings_df` como CSV

In [16]:
from google.colab import files

# Convertir bookings_df a CSV
bookings_df.to_csv('bookings.csv', index=False)

# Descargar el archivo
files.download('bookings.csv')

print("Archivo 'bookings.csv' generado y listo para descargar.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Archivo 'bookings.csv' generado y listo para descargar.


In [ ]:
import os
import numpy as np
import pandas as pd

os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/raw", exist_ok=True)
os.makedirs("reports/figures", exist_ok=True)
os.makedirs("reports/model_cards", exist_ok=True)

print("Folders ready ✅")

Folders ready ✅


In [ ]:
def make_levanteferries_dataset(
    start="2024-01-01",
    end="2025-12-31",
    seed=42,
    include_booking_curve=True
):
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start=start, end=end, freq="D")

    routes = [
        {"route_id": "DEN-IBZ", "origin": "Denia",     "dest": "Ibiza",      "distance_nm": 66},
        {"route_id": "DEN-PMI", "origin": "Denia",     "dest": "Palma",      "distance_nm": 130},
        {"route_id": "DEN-FOR", "origin": "Denia",     "dest": "Formentera", "distance_nm": 57},
        {"route_id": "VAL-IBZ", "origin": "Valencia",  "dest": "Ibiza",      "distance_nm": 95},
    ]

    dep_times = ["08:00", "12:00", "17:00", "20:00"]

    ships = [
        {"ship_type": "FAST", "capacity_pax": 900,  "capacity_veh": 220, "base_price": 55},
        {"ship_type": "CONV", "capacity_pax": 1200, "capacity_veh": 320, "base_price": 45},
    ]

    def is_holiday_proxy(d):
        fixed = {(1, 1), (1, 6), (5, 1), (8, 15), (10, 12), (11, 1), (12, 6), (12, 8), (12, 25)}
        if (d.month, d.day) in fixed:
            return 1
        if d.weekday() in (0, 4) and rng.random() < 0.03:
            return 1
        return 0

    def seasonality_multiplier(d):
        if d.month in (7, 8):  return 1.45
        if d.month in (6, 9):  return 1.20
        if d.month in (4, 5, 10): return 1.05
        return 0.90

    def weekday_multiplier(d):
        if d.weekday() == 5: return 1.25
        if d.weekday() == 6: return 1.15
        if d.weekday() == 4: return 1.10
        return 0.95

    def route_base_demand(route_id):
        return {"DEN-IBZ": 650, "DEN-PMI": 520, "DEN-FOR": 560, "VAL-IBZ": 420}[route_id]

    def time_multiplier(tstr):
        return {"08:00": 0.95, "12:00": 1.10, "17:00": 1.15, "20:00": 1.05}[tstr]

    def sea_state(d):
        base = 0.25 if d.month in (12, 1, 2) else 0.12
        return 1 if rng.random() < base else 0

    trips = []
    trip_id = 0

    for d in dates:
        holiday = is_holiday_proxy(d)
        season_mult = seasonality_multiplier(d)
        week_mult = weekday_multiplier(d)
        sea_bad = sea_state(d)

        for r in routes:
            freq = 2
            if r["route_id"] in ("DEN-IBZ", "DEN-FOR"):
                freq = 3
            if d.month in (7, 8):
                freq += 1
            if d.weekday() in (4, 5, 6):
                freq += 1

            chosen_times = rng.choice(dep_times, size=min(freq, len(dep_times)), replace=False)

            for tstr in chosen_times:
                ship = ships[0] if (r["route_id"] in ("DEN-IBZ", "DEN-FOR") and rng.random() < 0.60) else ships[1]

                base = route_base_demand(r["route_id"])
                latent = base * season_mult * week_mult * time_multiplier(tstr)

                latent *= (1.20 if holiday else 1.00)
                latent *= (0.88 if sea_bad else 1.00)
                latent *= rng.normal(1.0, 0.10)

                price = ship["base_price"] * (1.10 if d.month in (7, 8) else 1.00)
                price *= (1.05 if d.weekday() in (4, 5, 6) else 1.00)
                price *= (1.06 if holiday else 1.00)
                price *= rng.normal(1.0, 0.03)

                price_index = price / ship["base_price"]
                latent *= (1.0 - 0.12 * (price_index - 1.0))

                pax_latent = max(0, latent)
                veh_latent = max(0, (latent / 3.2) * rng.normal(1.0, 0.10))

                pax_real = int(min(ship["capacity_pax"], pax_latent))
                veh_real = int(min(ship["capacity_veh"], veh_latent))

                occ_pax = pax_real / ship["capacity_pax"]
                occ_veh = veh_real / ship["capacity_veh"]

                cancel_rate = np.clip(rng.normal(0.04, 0.015), 0.01, 0.10)
                if sea_bad:
                    cancel_rate = np.clip(cancel_rate + 0.03, 0.02, 0.18)

                no_show_rate = np.clip(rng.normal(0.015, 0.007), 0.0, 0.05)
                if sea_bad:
                    no_show_rate = np.clip(no_show_rate + 0.01, 0.0, 0.08)

                delay_minutes = int(max(0, rng.normal(12 if sea_bad else 6, 8)))
                revenue = pax_real * price + veh_real * (price * 0.65)

                trip_dt = pd.Timestamp(f"{d.date()} {tstr}")

                trips.append({
                    "company": "LevanteFerries",
                    "trip_id": f"T{trip_id:07d}",
                    "route_id": r["route_id"],
                    "origin_port": r["origin"],
                    "dest_port": r["dest"],
                    "departure_datetime_local": trip_dt,
                    "date": pd.Timestamp(d.date()),
                    "dep_time": tstr,
                    "weekday": trip_dt.weekday(),
                    "month": trip_dt.month,
                    "year": trip_dt.year,
                    "ship_type": ship["ship_type"],
                    "capacity_pax": ship["capacity_pax"],
                    "capacity_veh": ship["capacity_veh"],
                    "is_holiday_proxy": holiday,
                    "sea_bad_proxy": sea_bad,
                    "avg_ticket_price": float(price),
                    "price_index": float(price_index),
                    "pax_real": pax_real,
                    "veh_real": veh_real,
                    "occupancy_real_pax": float(occ_pax),
                    "occupancy_real_veh": float(occ_veh),
                    "cancel_rate": float(cancel_rate),
                    "no_show_rate": float(no_show_rate),
                    "delay_minutes": delay_minutes,
                    "revenue_real": float(revenue),
                })
                trip_id += 1

    trips_df = pd.DataFrame(trips).sort_values("departure_datetime_local").reset_index(drop=True)

    bookings_df = None
    if include_booking_curve:
        rows = []
        for _, tr in trips_df.iterrows():
            dep = tr["departure_datetime_local"]
            rng_h = int(np.clip(rng.normal(35, 10), 15, 60))
            start_day = (dep.normalize() - pd.Timedelta(days=rng_h))

            total_booked = int(np.clip(tr["pax_real"] / (1 - tr["cancel_rate"]), 0, tr["capacity_pax"]))
            days = pd.date_range(start_day, dep.normalize(), freq="D")

            k = np.linspace(0.8, 2.4, len(days))
            weights = np.exp(k)
            weights = weights / weights.sum()
            daily = rng.multinomial(total_booked, weights)

            canc_total = int(total_booked * tr["cancel_rate"])
            canc_days = min(len(days), 10)
            canc = np.zeros(len(days), dtype=int)
            if canc_total > 0:
                tail_w = np.linspace(1, 3, canc_days)
                tail_w = tail_w / tail_w.sum()
                canc_tail = rng.multinomial(canc_total, tail_w)
                canc[-canc_days:] = canc_tail

            cum = 0
            for dday, b, c in zip(days, daily, canc):
                cum += (b - c)
                rows.append({
                    "trip_id": tr["trip_id"],
                    "route_id": tr["route_id"],
                    "departure_date": dep.normalize(),
                    "booking_date": dday,
                    "bookings": int(b),
                    "cancellations": int(c),
                    "net_bookings": int(b - c),
                    "cum_net_bookings": int(max(0, cum)),
                    "days_to_departure": int((dep.normalize() - dday).days),
                })

        bookings_df = pd.DataFrame(rows)

    return trips_df, bookings_df

In [ ]:
trips_df, bookings_df = make_levanteferries_dataset(
    start="2024-01-01",
    end="2025-12-31",
    seed=42,
    include_booking_curve=True
)

print("Trips:", trips_df.shape)
print("Bookings:", bookings_df.shape)

trips_df.to_parquet(f"{PROJECT_ROOT}/data/processed/trips.parquet", index=False)
bookings_df.to_parquet(f"{PROJECT_ROOT}/data/processed/bookings.parquet", index=False)

display(trips_df.head())
display(bookings_df.head())

print("Saved ✅", f"{PROJECT_ROOT}/data/processed/")

import os
print("trips.parquet:", os.path.exists(f"{PROJECT_ROOT}/data/processed/trips.parquet"))
print("bookings.parquet:", os.path.exists(f"{PROJECT_ROOT}/data/processed/bookings.parquet"))

Trips: (8948, 26)
Bookings: (319231, 9)


,company,trip_id,route_id,origin_port,dest_port,departure_datetime_local,date,dep_time,weekday,month,...,avg_ticket_price,price_index,pax_real,veh_real,occupancy_real_pax,occupancy_real_veh,cancel_rate,no_show_rate,delay_minutes,revenue_real
0,LevanteFerries,T0000005,DEN-FOR,Denia,Formentera,2024-01-01 08:00:00,2024-01-01,08:00,0,1,...,48.477254,1.077272,477,139,0.397500,0.434375,0.043482,0.015817,7,27503.569920
1,LevanteFerries,T0000001,DEN-IBZ,Denia,Ibiza,2024-01-01 12:00:00,2024-01-01,12:00,0,1,...,49.313082,1.095846,729,238,0.607500,0.743750,0.027111,0.017581,0,43577.970711
2,LevanteFerries,T0000008,VAL-IBZ,Valencia,Ibiza,2024-01-01 12:00:00,2024-01-01,12:00,0,1,...,47.932900,1.065176,454,150,0.378333,0.468750,0.050668,0.020553,3,26434.994149
3,LevanteFerries,T0000000,DEN-IBZ,Denia,Ibiza,2024-01-01 17:00:00,2024-01-01,17:00,0,1,...,58.523593,1.064065,661,200,0.734444,0.909091,0.039748,0.009029,13,46292.161956
4,LevanteFerries,T0000004,DEN-PMI,Denia,Palma,2024-01-01 17:00:00,2024-01-01,17:00,0,1,...,46.535491,1.034122,579,192,0.482500,0.600000,0.056935,0.014202,0,32751.678725


,trip_id,route_id,departure_date,booking_date,bookings,cancellations,net_bookings,cum_net_bookings,days_to_departure
0,T0000005,DEN-FOR,2024-01-01,2023-11-27,4,0,4,4,35
1,T0000005,DEN-FOR,2024-01-01,2023-11-28,7,0,7,11,34
2,T0000005,DEN-FOR,2024-01-01,2023-11-29,9,0,9,20,33
3,T0000005,DEN-FOR,2024-01-01,2023-11-30,4,0,4,24,32
4,T0000005,DEN-FOR,2024-01-01,2023-12-01,3,0,3,27,31


Saved ✅ /content/drive/MyDrive/Colab Notebooks/Porfolio/Balearia/2 Demanda/data/processed/
trips.parquet: True
bookings.parquet: True


In [17]:
trips_df, bookings_df = make_levanteferries_dataset(
    start="2024-01-01",
    end="2025-12-31",
    seed=42,
    include_booking_curve=True
)

print("Trips:", trips_df.shape)
print("Bookings:", bookings_df.shape)

trips_df.to_parquet(f"{PROJECT_ROOT}/data/processed/trips.parquet", index=False)
bookings_df.to_parquet(f"{PROJECT_ROOT}/data/processed/bookings.parquet", index=False)

display(trips_df.head())
display(bookings_df.head())

print("Saved ✅", f"{PROJECT_ROOT}/data/processed/")

import os
print("trips.parquet:", os.path.exists(f"{PROJECT_ROOT}/data/processed/trips.parquet"))
print("bookings.parquet:", os.path.exists(f"{PROJECT_ROOT}/data/processed/bookings.parquet"))

Trips: (8948, 26)
Bookings: (319231, 9)


,company,trip_id,route_id,origin_port,dest_port,departure_datetime_local,date,dep_time,weekday,month,...,avg_ticket_price,price_index,pax_real,veh_real,occupancy_real_pax,occupancy_real_veh,cancel_rate,no_show_rate,delay_minutes,revenue_real
0,LevanteFerries,T0000005,DEN-FOR,Denia,Formentera,2024-01-01 08:00:00,2024-01-01,08:00,0,1,...,48.477254,1.077272,477,139,0.397500,0.434375,0.043482,0.015817,7,27503.569920
1,LevanteFerries,T0000001,DEN-IBZ,Denia,Ibiza,2024-01-01 12:00:00,2024-01-01,12:00,0,1,...,49.313082,1.095846,729,238,0.607500,0.743750,0.027111,0.017581,0,43577.970711
2,LevanteFerries,T0000008,VAL-IBZ,Valencia,Ibiza,2024-01-01 12:00:00,2024-01-01,12:00,0,1,...,47.932900,1.065176,454,150,0.378333,0.468750,0.050668,0.020553,3,26434.994149
3,LevanteFerries,T0000000,DEN-IBZ,Denia,Ibiza,2024-01-01 17:00:00,2024-01-01,17:00,0,1,...,58.523593,1.064065,661,200,0.734444,0.909091,0.039748,0.009029,13,46292.161956
4,LevanteFerries,T0000004,DEN-PMI,Denia,Palma,2024-01-01 17:00:00,2024-01-01,17:00,0,1,...,46.535491,1.034122,579,192,0.482500,0.600000,0.056935,0.014202,0,32751.678725


,trip_id,route_id,departure_date,booking_date,bookings,cancellations,net_bookings,cum_net_bookings,days_to_departure
0,T0000005,DEN-FOR,2024-01-01,2023-11-27,4,0,4,4,35
1,T0000005,DEN-FOR,2024-01-01,2023-11-28,7,0,7,11,34
2,T0000005,DEN-FOR,2024-01-01,2023-11-29,9,0,9,20,33
3,T0000005,DEN-FOR,2024-01-01,2023-11-30,4,0,4,24,32
4,T0000005,DEN-FOR,2024-01-01,2023-12-01,3,0,3,27,31


Saved ✅ /content/drive/MyDrive/Colab Notebooks/Porfolio/Balearia/2 Demanda/data/processed/
trips.parquet: True
bookings.parquet: True
